In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
['DFU', 'Documents', 'Pictures']

['DFU', 'Documents', 'Pictures']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.losses import CategoricalCrossentropy

from sklearn.utils.class_weight import compute_class_weight

# =====================================================
# PATH
# =====================================================

DATASET_PATH = "/content/drive/MyDrive/DFU"

BEST_MODEL_PATH = "/content/drive/MyDrive/DFU/best_dfu_model.keras"
FINAL_MODEL_PATH = "/content/drive/MyDrive/DFU/dfu_final_model.keras"

# =====================================================
# CONFIG
# =====================================================

IMG_SIZE = 260
BATCH_SIZE = 32

EPOCHS_STAGE1 = 12
EPOCHS_STAGE2 = 15

# =====================================================
# DATA AUGMENTATION
# =====================================================

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.95,1.05]
)

valid_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# =====================================================
# LOAD DATA
# =====================================================

train_data = train_gen.flow_from_directory(
    os.path.join(DATASET_PATH,'train'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_data = valid_gen.flow_from_directory(
    os.path.join(DATASET_PATH,'valid'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_data.class_indices)

# =====================================================
# CLASS WEIGHT
# =====================================================

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(weights))

print(class_weights)

# =====================================================
# MODEL
# =====================================================

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

base_model.trainable = False

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.4)(x)

outputs = Dense(
    4,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=outputs
)

# =====================================================
# COMPILE STAGE 1
# =====================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=CategoricalCrossentropy(
        label_smoothing=0.05
    ),
    metrics=['accuracy']
)

callbacks = [

    ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# =====================================================
# STAGE 1 TRAIN
# =====================================================

history1 = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks,
    class_weight=class_weights
)

# =====================================================
# LOAD BEST MODEL
# =====================================================

model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

# =====================================================
# FINE TUNING
# =====================================================

base_model = model.layers[1]

base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=CategoricalCrossentropy(
        label_smoothing=0.05
    ),
    metrics=['accuracy']
)

history2 = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks,
    class_weight=class_weights
)

# =====================================================
# SAVE FINAL MODEL
# =====================================================

model.save(FINAL_MODEL_PATH)

print("\nTraining Complete")

# =====================================================
# EVALUATE
# =====================================================

loss, acc = model.evaluate(valid_data)

print("\nValidation Accuracy =", acc)

print("\nSaved Files")
print(BEST_MODEL_PATH)
print(FINAL_MODEL_PATH)

# =====================================================
# DOWNLOAD FINAL MODEL
# =====================================================

from google.colab import files

files.download(FINAL_MODEL_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 9639 images belonging to 4 classes.
Found 282 images belonging to 4 classes.
Classes: {'Grade 1': 0, 'Grade 2': 1, 'Grade 3': 2, 'Grade 4': 3}
{0: np.float64(1.07578125), 1: np.float64(1.0276119402985076), 2: np.float64(0.8781887755102041), 3: np.float64(1.0431818181818182)}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/12
302/302 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.4806 - loss: 1.4715
Epoch 1: val_accuracy improved from None to 0.58865, saving model to /content/drive/MyDrive/DFU/best_dfu_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DFU/best_dfu_model.keras
302/302 ━━━━━━━━━━━━━━━━━━━━ 2880s 9s/step - accuracy: 0.5477 - loss: 1.2556 - val_accuracy: 0.5887 - val_loss: 1.0709 - learning_rate: 0.0010
Epoch 2/12
302/302 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6599 - loss: 0.9258
Epoch 2: val_accuracy did

AttributeError: 'Rescaling' object has no attribute 'layers'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils.class_weight import compute_class_weight

# =====================================================
# PATH
# =====================================================

DATASET_PATH = "/content/drive/MyDrive/DFU"

BEST_MODEL_PATH = "/content/drive/MyDrive/DFU/best_dfu_model.keras"

FINETUNE_BEST_PATH = "/content/drive/MyDrive/DFU/best_dfu_finetune.keras"

FINAL_MODEL_PATH = "/content/drive/MyDrive/DFU/dfu_final_model_v2.keras"

# =====================================================
# CONFIG
# =====================================================

IMG_SIZE = 260
BATCH_SIZE = 32

# =====================================================
# DATA
# =====================================================

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

valid_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_data = valid_gen.flow_from_directory(
    os.path.join(DATASET_PATH, "valid"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_data.class_indices)

# =====================================================
# CLASS WEIGHT
# =====================================================

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(weights))

print("Class Weights:", class_weights)

# =====================================================
# LOAD BEST MODEL
# =====================================================

model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

print("Best Model Loaded")

# =====================================================
# FIND EFFICIENTNET
# =====================================================

base_model = None

for layer in model.layers:
    if "efficientnet" in layer.name.lower():
        base_model = layer
        break

print("Base Model:", base_model.name)

# =====================================================
# FINE TUNE
# =====================================================

base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

# =====================================================
# COMPILE
# =====================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=CategoricalCrossentropy(
        label_smoothing=0.03
    ),
    metrics=['accuracy']
)

# =====================================================
# CALLBACKS
# =====================================================

callbacks = [

    ModelCheckpoint(
        FINETUNE_BEST_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# =====================================================
# TRAIN
# =====================================================

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

# =====================================================
# SAVE FINAL MODEL
# =====================================================

model.save(FINAL_MODEL_PATH)

print("Training Complete")

# =====================================================
# EVALUATE
# =====================================================

loss, acc = model.evaluate(valid_data)

print("\nValidation Accuracy =", acc)

# =====================================================
# SHOW FILES
# =====================================================

print("\nSaved Files")

print(FINETUNE_BEST_PATH)
print(FINAL_MODEL_PATH)

# =====================================================
# DOWNLOAD
# =====================================================

from google.colab import files

files.download(FINAL_MODEL_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 9639 images belonging to 4 classes.
Found 282 images belonging to 4 classes.
Classes: {'Grade 1': 0, 'Grade 2': 1, 'Grade 3': 2, 'Grade 4': 3}
Class Weights: {0: np.float64(1.07578125), 1: np.float64(1.0276119402985076), 2: np.float64(0.8781887755102041), 3: np.float64(1.0431818181818182)}
Best Model Loaded


AttributeError: 'NoneType' object has no attribute 'name'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.losses import CategoricalCrossentropy

from sklearn.utils.class_weight import compute_class_weight

# =====================================================
# PATH
# =====================================================

DATASET_PATH = "/content/drive/MyDrive/DFU"

BEST_MODEL = "/content/drive/MyDrive/DFU/best_dfu_model.keras"

NEW_BEST_MODEL = "/content/drive/MyDrive/DFU/best_dfu_model_finetune.keras"

FINAL_MODEL = "/content/drive/MyDrive/DFU/dfu_final_model.keras"

# =====================================================
# CONFIG
# =====================================================

IMG_SIZE = 260
BATCH_SIZE = 32

FINE_TUNE_EPOCHS = 15

# =====================================================
# DATA
# =====================================================

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.95,1.05]
)

valid_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_gen.flow_from_directory(
    os.path.join(DATASET_PATH,'train'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_data = valid_gen.flow_from_directory(
    os.path.join(DATASET_PATH,'valid'),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_data.class_indices)

# =====================================================
# CLASS WEIGHT
# =====================================================

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(weights))

print("Class Weights:", class_weights)

# =====================================================
# LOAD MODEL
# =====================================================

model = tf.keras.models.load_model(BEST_MODEL)

print("\nBest Model Loaded")

# =====================================================
# FIND EFFICIENTNET
# =====================================================

base_model = None

for layer in model.layers:

    if "efficientnet" in layer.name.lower():

        base_model = layer
        break

if base_model is None:

    print("\nModel Layers")

    for layer in model.layers:
        print(layer.name)

    raise Exception(
        "ไม่พบ EfficientNet Layer"
    )

print("\nFound Base Model =", base_model.name)

# =====================================================
# FINE TUNE
# =====================================================

base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

print("\nTrainable Layers")

count = 0

for layer in base_model.layers:

    if layer.trainable:
        count += 1

print(count)

# =====================================================
# COMPILE
# =====================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=CategoricalCrossentropy(
        label_smoothing=0.05
    ),
    metrics=['accuracy']
)

# =====================================================
# CALLBACKS
# =====================================================

callbacks = [

    ModelCheckpoint(
        NEW_BEST_MODEL,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# =====================================================
# TRAIN
# =====================================================

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights
)

# =====================================================
# SAVE FINAL
# =====================================================

model.save(FINAL_MODEL)

print("\nSaved")

print(NEW_BEST_MODEL)
print(FINAL_MODEL)

# =====================================================
# EVALUATE
# =====================================================

loss, acc = model.evaluate(valid_data)

print("\nValidation Accuracy =", acc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 9639 images belonging to 4 classes.
Found 282 images belonging to 4 classes.
Classes: {'Grade 1': 0, 'Grade 2': 1, 'Grade 3': 2, 'Grade 4': 3}
Class Weights: {0: np.float64(1.07578125), 1: np.float64(1.0276119402985076), 2: np.float64(0.8781887755102041), 3: np.float64(1.0431818181818182)}

Best Model Loaded

Model Layers
input_layer
rescaling
normalization
rescaling_1
stem_conv_pad
stem_conv
stem_bn
stem_activation
block1a_dwconv
block1a_bn
block1a_activation
block1a_se_squeeze
block1a_se_reshape
block1a_se_reduce
block1a_se_expand
block1a_se_excite
block1a_project_conv
block1a_project_bn
block2a_expand_conv
block2a_expand_bn
block2a_expand_activation
block2a_dwconv_pad
block2a_dwconv
block2a_bn
block2a_activation
block2a_se_squeeze
block2a_se_reshape
block2a_se_reduce
block2a_se_expand
block2a_se_excite
block2a_project_conv
block2a_project_bn
block2b_

Exception: ไม่พบ EfficientNet Layer